# Validation Test Evaluation (comparison between XDG and WNLT) - Oscillating droplet

This notebook and the corresponding simulation setup notebook (OscillatingDroplet_DACH_Comparison_Run.ipynb) are part of published results (cf. V.A.) found in *From weakly to strongly nonlinear viscous drop shape oscillations: An analytical and numerical study* (https://doi.org/10.1103/PhysRevFluids.9.063601). 

In [ ]:
#r "BoSSSpad.dll"
//#r "../../../src/L4-application/BoSSSpad/bin/Release/net8.0/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
BoSSSshell.WorkflowMgm.Init("OscillatingDropletPaper");

Opening existing database 'C:\BoSSS-localJobs\OscillatingDropletPaper_OnJenkins'.


In [3]:
using System.IO;
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.XNSECommon;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

In [5]:
bool runShortTests = true; // set to 'false' to run all test cases for the whole period of time (up tp 7.0), otherwise they will run only up to 0.5

## Sessions

In [ ]:
static var sessions = BoSSSshell.WorkflowMgm.Sessions.ToList();
//static var sessions = databases.Pick(0).Sessions;
sessions

#0: OscillatingDropletPaper	OD3D_m2_Oh01_eta06_stagnantInit*	12/6/2025 10:30:45 PM	10651a2d...
#1: OscillatingDropletPaper	OD3D_m4_Oh01_eta02_stagnantInit*	12/6/2025 10:31:35 PM	b2ad17f9...
#2: OscillatingDropletPaper	OD3D_m4_Oh01_eta03_stagnantInit*	12/6/2025 10:31:48 PM	1a466c4b...
#3: OscillatingDropletPaper	OD3D_m2_Oh01_eta07_stagnantInit*	12/6/2025 10:30:57 PM	9b83588b...
#4: OscillatingDropletPaper	OD3D_m2_Oh01_eta03_stagnantInit*	12/6/2025 10:30:22 PM	42c06632...
#5: OscillatingDropletPaper	OD3D_m4_Oh01_eta06_stagnantInit*	12/6/2025 10:32:15 PM	f20038ae...
#6: OscillatingDropletPaper	OD3D_m3_Oh01_eta05_stagnantInit*	12/6/2025 10:31:22 PM	5bd0a0e3...
#7: OscillatingDropletPaper	OD3D_m2_Oh01_eta05_stagnantInit*	12/6/2025 10:30:33 PM	abb9c251...
#8: OscillatingDropletPaper	OD3D_m4_Oh01_eta07_stagnantInit*	12/6/2025 10:32:31 PM	e1f7fb46...
#9: OscillatingDropletPaper	OD3D_m3_Oh01_eta03_stagnantInit*	12/6/2025 10:31:10 PM	ca424e65...
#10: OscillatingDropletPaper	OD3D_m4_Oh01_eta05_st

## Comparison between XDG and WNLT

In [6]:
string[] mOhS = { "m2_Oh01", "m3_Oh01", "m4_Oh01", "m4_Oh056" };
List<string>[] studyCases = new List<string>[mOhS.Length];
studyCases[0] = new List<string>() { "eta01", "eta02", "eta04" };
studyCases[1] = new List<string>() { "eta015", "eta04" };
studyCases[2] = new List<string>() { "eta01", "eta04" };
studyCases[3] = new List<string>() { "eta005" };
string[] initS = { "stagnantInit", "thirdOrderInit" };

In [ ]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Dez12_000000.OscillatingDropletPaper");

Opening existing database '\\dc3\userspace\smuda\Databases\OscillatingDropletPaper'.


In [21]:
List<ISessionInfo> evalSessOrigin = new List<ISessionInfo>();

In [22]:
for (int i = 0; i < mOhS.Length; i++) {
foreach (string etaCase in studyCases[i]) {
foreach (string init in initS) {

    if (mOhS[i] == "m3_Oh01" && etaCase == "eta04" && init == "thirdOrderInit" 
        || mOhS[i] == "m4_Oh01" && etaCase == "eta04" && init == "thirdOrderInit" )
        continue; 

    string studyName = $"OD3D_{mOhS[i]}_{etaCase}_{init}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));

    int studyCount = studySess.Count();
    if (studyCount == 0) {
        Console.WriteLine("Not found in orignDB!");
    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSessOrigin.Add(studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSessOrigin.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
}
}
evalSessOrigin

Searching for: OD3D_m2_Oh01_eta01_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m2_Oh01_eta01_thirdOrderInit
Restart session found! Take last run
Searching for: OD3D_m2_Oh01_eta02_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m2_Oh01_eta02_thirdOrderInit
Restart session found! Take last run
Searching for: OD3D_m2_Oh01_eta04_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m2_Oh01_eta04_thirdOrderInit
Restart session found! Take last run
Searching for: OD3D_m3_Oh01_eta015_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m3_Oh01_eta015_thirdOrderInit
Restart session found! Take last run
Searching for: OD3D_m3_Oh01_eta04_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m4_Oh01_eta01_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m4_Oh01_eta01_thirdOrderInit
Restart session found! Take last run
Searching for: OD3D_m4_Oh01_eta04_stagnantInit
Restart session f

#0: OscillatingDropletPaper	OD3D_m2_Oh01_eta01_stagnantInit_restart3	6/15/2022 8:37:41 AM	dd97e2f1...
#1: OscillatingDropletPaper	OD3D_m2_Oh01_eta01_thirdOrderInit_restart3	6/15/2022 8:37:38 AM	0d35c433...
#2: OscillatingDropletPaper	OD3D_m2_Oh01_eta02_stagnantInit_restart3	6/15/2022 8:37:47 AM	95fc59fd...
#3: OscillatingDropletPaper	OD3D_m2_Oh01_eta02_thirdOrderInit_restart3	6/15/2022 9:32:54 AM	d8e3f635...
#4: OscillatingDropletPaper	OD3D_m2_Oh01_eta04_stagnantInit_restart3	6/13/2022 3:31:38 PM	ceb7f024...
#5: OscillatingDropletPaper	OD3D_m2_Oh01_eta04_thirdOrderInit_restart4	1/19/2023 11:08:11 AM	9d96318d...
#6: OscillatingDropletPaper	OD3D_m3_Oh01_eta015_stagnantInit_restart3	3/23/2023 2:36:57 PM	d0b3ed73...
#7: OscillatingDropletPaper	OD3D_m3_Oh01_eta015_thirdOrderInit_restart3	3/23/2023 2:37:51 PM	dc3aed05...
#8: OscillatingDropletPaper	OD3D_m3_Oh01_eta04_stagnantInit_restart2	7/11/2022 1:38:30 PM	b97b80ef...
#9: OscillatingDropletPaper	OD3D_m4_Oh01_eta01_stagnantInit_restart3	6/

In [23]:
NUnit.Framework.Assert.AreEqual(14, evalSessOrigin.Count, $"Not enough sessions for evaluation.");

In [13]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [14]:
for (int i = 0; i < mOhS.Length; i++) {
foreach (string etaCase in studyCases[i]) {
foreach (string init in initS) {
    
    if (mOhS[i] == "m3_Oh01" && etaCase == "eta04" && init == "thirdOrderInit" 
        || mOhS[i] == "m4_Oh01" && etaCase == "eta04" && init == "thirdOrderInit" )
        continue; 

    string studyName = $"OD3D_{mOhS[i]}_{etaCase}_{init}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = sessions.Where(sess => sess.Name.Contains(studyName));

    int studyCount = studySess.Count();
    if (studyCount == 0) {
        //Console.WriteLine("Not found in wmg! study will be excluded from check against origin session");
        
        // try to get from originDB
        var originSess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));
        int originCount = originSess.Count();
        if (originCount == 0) {
            Console.WriteLine("Not found!");
        } else if (originCount > 1) {
            Console.WriteLine("Restart session found in originDB! Take last run");       
            evalSess.Add(originSess.Where(sess => sess.Name.EndsWith($"_restart{originCount-1}")).Single());
        } else {
            evalSess.Add(originSess.Single());
            Console.WriteLine("found in originDB");
        }
        
    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
}
}
evalSess

Searching for: OD3D_m2_Oh01_eta01_stagnantInit
found
Searching for: OD3D_m2_Oh01_eta01_thirdOrderInit
found
Searching for: OD3D_m2_Oh01_eta02_stagnantInit
found
Searching for: OD3D_m2_Oh01_eta02_thirdOrderInit
found
Searching for: OD3D_m2_Oh01_eta04_stagnantInit
found
Searching for: OD3D_m2_Oh01_eta04_thirdOrderInit
found
Searching for: OD3D_m3_Oh01_eta015_stagnantInit
found
Searching for: OD3D_m3_Oh01_eta015_thirdOrderInit
found
Searching for: OD3D_m3_Oh01_eta04_stagnantInit
found
Searching for: OD3D_m4_Oh01_eta01_stagnantInit
found
Searching for: OD3D_m4_Oh01_eta01_thirdOrderInit
found
Searching for: OD3D_m4_Oh01_eta04_stagnantInit
found
Searching for: OD3D_m4_Oh056_eta005_stagnantInit
found
Searching for: OD3D_m4_Oh056_eta005_thirdOrderInit
found


#0: OscillatingDropletPaper	OD3D_m2_Oh01_eta01_stagnantInit	12/4/2025 3:10:23 PM	90609365...
#1: OscillatingDropletPaper	OD3D_m2_Oh01_eta01_thirdOrderInit	12/4/2025 3:10:12 PM	1959df4c...
#2: OscillatingDropletPaper	OD3D_m2_Oh01_eta02_stagnantInit	12/4/2025 3:10:49 PM	0e7cdee1...
#3: OscillatingDropletPaper	OD3D_m2_Oh01_eta02_thirdOrderInit	12/4/2025 3:10:36 PM	8af0ab80...
#4: OscillatingDropletPaper	OD3D_m2_Oh01_eta04_stagnantInit	12/4/2025 3:11:22 PM	012d30cd...
#5: OscillatingDropletPaper	OD3D_m2_Oh01_eta04_thirdOrderInit	12/4/2025 3:11:08 PM	339fc22a...
#6: OscillatingDropletPaper	OD3D_m3_Oh01_eta015_stagnantInit	12/4/2025 3:11:58 PM	36400c0b...
#7: OscillatingDropletPaper	OD3D_m3_Oh01_eta015_thirdOrderInit	12/4/2025 3:11:36 PM	7f898d75...
#8: OscillatingDropletPaper	OD3D_m3_Oh01_eta04_stagnantInit	12/4/2025 3:12:10 PM	87476317...
#9: OscillatingDropletPaper	OD3D_m4_Oh01_eta01_stagnantInit	12/4/2025 3:12:42 PM	e955f0a0...
#10: OscillatingDropletPaper	OD3D_m4_Oh01_eta01_thirdOrderIn

In [15]:
NUnit.Framework.Assert.AreEqual(14, evalSess.Count, $"Not enough sessions for evaluation.");

In [16]:
sessions.AddRange(originDB.Sessions);
sessions

#0: OscillatingDropletPaper	OD3D_m2_Oh01_eta06_stagnantInit*	12/6/2025 10:30:45 PM	10651a2d...
#1: OscillatingDropletPaper	OD3D_m4_Oh01_eta02_stagnantInit*	12/6/2025 10:31:35 PM	b2ad17f9...
#2: OscillatingDropletPaper	OD3D_m4_Oh01_eta03_stagnantInit*	12/6/2025 10:31:48 PM	1a466c4b...
#3: OscillatingDropletPaper	OD3D_m2_Oh01_eta07_stagnantInit*	12/6/2025 10:30:57 PM	9b83588b...
#4: OscillatingDropletPaper	OD3D_m2_Oh01_eta03_stagnantInit*	12/6/2025 10:30:22 PM	42c06632...
#5: OscillatingDropletPaper	OD3D_m4_Oh01_eta06_stagnantInit*	12/6/2025 10:32:15 PM	f20038ae...
#6: OscillatingDropletPaper	OD3D_m3_Oh01_eta05_stagnantInit*	12/6/2025 10:31:22 PM	5bd0a0e3...
#7: OscillatingDropletPaper	OD3D_m2_Oh01_eta05_stagnantInit*	12/6/2025 10:30:33 PM	abb9c251...
#8: OscillatingDropletPaper	OD3D_m4_Oh01_eta07_stagnantInit*	12/6/2025 10:32:31 PM	e1f7fb46...
#9: OscillatingDropletPaper	OD3D_m3_Oh01_eta03_stagnantInit*	12/6/2025 10:31:10 PM	ca424e65...
#10: OscillatingDropletPaper	OD3D_m4_Oh01_eta05_st

### compute times

In [17]:
foreach (var sess in evalSess) {
    var nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string sessName = "";
    int nameLength = 5;
    for (int i = 0; i < nameLength; i++) {
        sessName += nameData[i];
        if (i != nameLength - 1)
            sessName += "_";
    }

    // determine compute time
    Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
    procSIs.Push(sess);
    var currSI = sess;
    var currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
    int timesteps = currTimesteps.Last().TimeStepNumber.MajorNumber;
    double physicalTime = currTimesteps.Last().PhysicalTime;
    TimeSpan computeTime = currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime;
    var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    while(!rSIs.IsNullOrEmpty()) {
        var rSI = rSIs.Single();
        procSIs.Push(rSI);
        currSI = rSI;
        currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
        computeTime = computeTime + (currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime);
        rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    }

    Console.WriteLine($"Session - {sessName}: timesteps = {timesteps} (t_end = {physicalTime}); compute time = {computeTime.Days}:{computeTime.Hours}");
}

Session - OD3D_m2_Oh01_eta01_stagnantInit: timesteps = 20 (t_end = 0.10000000000000002); compute time = 0:11
Session - OD3D_m2_Oh01_eta01_thirdOrderInit: timesteps = 20 (t_end = 0.10000000000000002); compute time = 0:11
Session - OD3D_m2_Oh01_eta02_stagnantInit: timesteps = 20 (t_end = 0.10000000000000002); compute time = 0:10
Session - OD3D_m2_Oh01_eta02_thirdOrderInit: timesteps = 20 (t_end = 0.10000000000000002); compute time = 0:11
Session - OD3D_m2_Oh01_eta04_stagnantInit: timesteps = 20 (t_end = 0.10000000000000002); compute time = 0:11
Session - OD3D_m2_Oh01_eta04_thirdOrderInit: timesteps = 20 (t_end = 0.10000000000000002); compute time = 1:2
Session - OD3D_m3_Oh01_eta015_stagnantInit: timesteps = 20 (t_end = 0.10000000000000002); compute time = 0:11
Session - OD3D_m3_Oh01_eta015_thirdOrderInit: timesteps = 20 (t_end = 0.10000000000000002); compute time = 0:12
Session - OD3D_m3_Oh01_eta04_stagnantInit: timesteps = 20 (t_end = 0.10000000000000002); compute time = 0:13
Session - 

### read log data

In [18]:
public static List<Plot2Ddata> ReadLogDataForSphericalHarmonics(this List<ISessionInfo> sessions) {

    List<Plot2Ddata> plotData = new List<Plot2Ddata>();

    int numberSessions = sessions.Count();

    // Read all data
    for(int j = 0; j < numberSessions; j++) {

        var sess = sessions.ElementAt(j);
        Console.WriteLine("Processing session: " + sess.Name);

        // Get session history
        List<Guid> restartSessionList = new List<Guid>();
        restartSessionList.Add(sess.ID);
        Guid restartID = sess.RestartedFrom;

        var sessionIDs = new List<Guid>();
        var directories = Directory.GetDirectories(Path.Combine(sess.Database.Path, "sessions"));
        foreach(string dirname in directories) {
            string IDname = dirname.Split(new string[] { "\\" }, StringSplitOptions.RemoveEmptyEntries).Last();
            sessionIDs.Add(new Guid(IDname));
        }

        while(restartID != Guid.Empty) {
            try {
                var rSess = sessionIDs.Where(sess => sess == restartID).Single();
                Console.WriteLine("  Found restart session: " + rSess);
                restartSessionList.Add(rSess);

                string path_obj = Path.Combine(sess.Database.Path, "sessions", restartID.ToString(), "Control-obj.txt");
                string ctrlfileContent = File.ReadAllText(path_obj);
                var ctrl = (XNSE_Control)AppControl.Deserialize(ctrlfileContent);

                if(ctrl.RestartInfo != null) {
                    restartID = ctrl.RestartInfo.Item1;
                } else {
                    restartID = Guid.Empty;
                }

            } catch { // do nothing if session not found
                restartID = Guid.Empty;
            }
        }
        restartSessionList.Reverse();

        Console.WriteLine("  Merging data from " + restartSessionList.Count + " sessions.");

        string path = Path.Combine(sess.Database.Path, "sessions", sess.ID.ToString(), "SphericalHarmonics_NEW.txt");   // there might be some updated data in the original files
        string[] lines;
        try {
            lines = File.ReadAllLines(path);
        } catch {
            path = Path.Combine(sess.Database.Path, "sessions", sess.ID.ToString(), "SphericalHarmonics.txt");
            lines = File.ReadAllLines(path);
        }
        string[] modes = lines[0].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries).Skip(1).ToArray();

        Console.WriteLine("  Found modes: " + string.Join(", ", modes));

        List<double> time = new List<double>();
        int numModes = modes.Length;
        List<double>[] modesData = new List<double>[numModes];
        for(int m = 0; m < numModes; m++) {
            modesData[m] = new List<double>();
        }

        for(int rSi = 0; rSi < restartSessionList.Count(); rSi++) {

            path = Path.Combine(sess.Database.Path, "sessions", restartSessionList.ElementAt(rSi).ToString(), "SphericalHarmonics_NEW.txt");
            try {
                lines = File.ReadAllLines(path);
            } catch {
                path = Path.Combine(sess.Database.Path, "sessions", restartSessionList.ElementAt(rSi).ToString(), "SphericalHarmonics.txt");
                lines = File.ReadAllLines(path);
            }
            lines = File.ReadAllLines(path);

            // skip SetUpData
            int line0 = 0;
            for(int i = 0; i < lines.Length; i++) {
                string check = lines[i].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[0];
                if(check.Equals("time") || check.Contains("#")) {
                    line0 = i + 1;
                    break;
                }
            }

            for(int i = 0; i < lines.Length - line0; i++) {

                double l_time = Convert.ToDouble(lines[line0 + i].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[0]);
                double[] l_values = new double[numModes];
                for(int m = 0; m < numModes; m++) {
                    l_values[m] = Convert.ToDouble(lines[line0 + i].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[m + 1]);
                }

                if(time.IsNullOrEmpty() || l_time > time.Last()) {
                    time.Add(l_time);
                    for(int m = 0; m < numModes; m++) {
                        modesData[m].Add(l_values[m]);
                    }
                }
            }

        }

        // Build DataSets
        Console.WriteLine("Build DataSets");
    
        KeyValuePair<string, double[][]>[] dataRowsValue = new KeyValuePair<string, double[][]>[numModes];
        for(int m = 0; m < numModes; m++) {
            dataRowsValue[m] = new KeyValuePair<string, double[][]>(modes[m], new double[][] { time.ToArray(), modesData[m].ToArray() });
        }          
        Plot2Ddata plt = new Plot2Ddata(dataRowsValue);
        plt.Title = sess.Name.Replace("_", "-");
        plt.Xlabel = "time";
        plt.Ylabel = "a_m";
        
        plotData.Add(plt);      

        Console.WriteLine("... Done"); 

    }

    return plotData;
}

In [19]:
var modeData = evalSess.ReadLogDataForSphericalHarmonics();

Processing session: OD3D_m2_Oh01_eta01_stagnantInit
  Merging data from 1 sessions.
  Found modes: (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0)
Build DataSets
... Done
Processing session: OD3D_m2_Oh01_eta01_thirdOrderInit
  Merging data from 1 sessions.
  Found modes: (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0)
Build DataSets
... Done
Processing session: OD3D_m2_Oh01_eta02_stagnantInit
  Merging data from 1 sessions.
  Found modes: (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0)
Build DataSets
... Done
Processing session: OD3D_m2_Oh01_eta02_thirdOrderInit
  Merging data from 1 sessions.
  Found modes: (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0)
Build DataSets
... Done
Processing session: OD3D_m2_Oh01_eta04_stagnantInit
  Merging data from 1 sessions.
  Found modes: (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0)
Build DataSets
... Done
Processing session: OD3D_m

In [24]:
var modeDataOrigin = evalSessOrigin.ReadLogDataForSphericalHarmonics();

Processing session: OD3D_m2_Oh01_eta01_stagnantInit_restart3
  Found restart session: a2d5a18a-39f4-449a-8013-54106406c1cf
  Found restart session: 2a51c1d4-cf2c-45f6-9baf-be68918ddc54
  Found restart session: 59bb73dc-0683-42a1-a615-e87ed3524000
  Merging data from 4 sessions.
  Found modes: (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0)
Build DataSets
... Done
Processing session: OD3D_m2_Oh01_eta01_thirdOrderInit_restart3
  Found restart session: bc8c9ffb-fb8d-4976-a321-a406fcd0f6ac
  Found restart session: 67803604-6f99-4143-b101-522f73997b65
  Found restart session: 51afe8ea-2449-4767-8122-6346514a6e04
  Merging data from 4 sessions.
  Found modes: (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0)
Build DataSets
... Done
Processing session: OD3D_m2_Oh01_eta02_stagnantInit_restart3
  Found restart session: 33d67916-8020-4464-a16e-f55562a81bb0
  Found restart session: 6b199ada-b490-49b1-99d1-0a7ec09620fd
  Found restart session: 2e275baf-7

### aspect ratio L/W over time - by mode decomposition

In [25]:
public static double[] GetAspectRatioByModeDecomposition(Plot2Ddata modeData, int reqMaxL = -1) {

    int numTimesteps = modeData.dataGroups[0].Abscissas.Length;
    double[] aspectRatio = new double[numTimesteps];

    for (int ts = 0; ts < numTimesteps; ts++) {

        double northPole = 0.0;
        double southPole = 0.0;
        double equatorWidth = 0.0;

        int maxL = modeData.dataGroups.Count();
        if (reqMaxL >= 0) {
            maxL = Math.Min(maxL, reqMaxL);
        }
        for (int m = 0; m < maxL; m++) {

            double mValue = modeData.dataGroups[m].Values[ts];

            northPole += mValue * SphericalHarmonics.MyLegendre(m,0,Math.Cos(0.0));
            southPole += mValue * SphericalHarmonics.MyLegendre(m,0,Math.Cos(Math.PI));
            equatorWidth += mValue * SphericalHarmonics.MyLegendre(m,0,Math.Cos(Math.PI/2.0));   
        }
        aspectRatio[ts] = (northPole + southPole)/(2.0 * equatorWidth);      
    }

    return aspectRatio;
}

In [26]:
public static LineColors GetCustomLineColorFromIndex(int idx, LineColors[] colors = null) {

    if (colors != null) {
        if (idx < colors.Length)
            return colors[idx];
        else 
            throw new IndexOutOfRangeException("Color index out of range of custom color array.");
    } else {
        return (LineColors)(idx);
    }
}

In [27]:
public static Plot2Ddata GetAspectRatioOverTimeByModeDecomposition_Plot2Ddata(List<Plot2Ddata> modeDataS, string[] studies, double tend = 7.0) {

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "t";
    plt.Ylabel = "L/W";

    int lcIdx = 0;
    for (int i = 0; i < studies.Length; i++) {
        var datPlt = modeDataS.Where(plt => plt.Title.Contains(studies[i])).Single();

        double[] aspectRatio = GetAspectRatioByModeDecomposition(datPlt);
    
        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines;
        fmt.DashType = studies[i].Contains("stagnantInit") ? DashTypes.Solid : DashTypes.DotDashed;
        fmt.LineWidth = 2;
        LineColors[] customColors = new LineColors[] { LineColors.Blue, LineColors.Red, LineColors.Green };
        fmt.LineColor = GetCustomLineColorFromIndex(lcIdx, customColors);

        string[] nameParts = datPlt.Title.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        plt.AddDataGroup($"XDG - {nameParts[4]}", datPlt.dataGroups[0].Abscissas, aspectRatio, fmt);

        lcIdx++;      
    }   

    plt.XrangeMin = 0.0;
    plt.XrangeMax = tend;

    plt.ShowLegend = true;

    return plt;        
}

In [31]:
public static MultidimensionalArray ReadPaperData(string dat, string localPath = null, int headerSize = 0) {

    string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
    //string[] nameData = dat.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);

    string dataPath = (userName.Equals(@"FDY\jenkinsci")) ? dat : localPath + dat;

    using(StreamReader txt = new StreamReader(dataPath)) {
        try {
            for (int i = 0; i < headerSize; i++) {
                txt.ReadLine(); // skip header
            }
            return IMatrixExtensions.LoadFromStream(txt);
        }   finally {
            txt.Close();
        }
    }
}

### FIG. 5.
Aspect ratio $L/W$ over time for $Oh = 0.1$ and $m = 2$. Left: $\eta_0 = 0.1$ (case 1.1). Right: $\eta_0 = 0.2$ (case 1.2).

In [34]:
List<Plot2Ddata> plotModeData = (runShortTests) ? modeDataOrigin : modeData;

In [119]:
public static void CheckAgainstOriginModeData(List<Plot2Ddata> StudyData, List<Plot2Ddata> OriginData, string study, double errThrsh, int maxL = 4, bool checkOddModes = false, bool output = false) {

    //foreach (var study in studies) {

    var Splt = StudyData.Where(grp => grp.Title.Contains(study)).Single();
    var Oplt = OriginData.Where(grp => grp.Title.Contains(study)).Single();

    //int mode = 0;
    foreach (var Sgrp in Splt.dataGroups) {    

        int mode = Int32.Parse(string.Concat(Sgrp.Name.Where( Char.IsDigit ).SkipLast(1)));
        
        if (mode > maxL)
            continue;

        if (!checkOddModes && mode % 2 == 1) {
            mode++;
            continue;  
        }

        Console.WriteLine($"checking mode {mode}:"); 

        var Ogrp = Oplt.dataGroups.Where(grp => Int32.Parse(string.Concat(grp.Name.Where( Char.IsDigit ).SkipLast(1))) == mode).Single();
        var errorData = ISessionInfoExtensions.ComputeErrorNormsForLogData(Sgrp, Ogrp);

        if (output) {
            Console.WriteLine($"    l1-error norm = {errorData["l_1 error norm"]}");
            Console.WriteLine($"    l2-error norm = {errorData["l_2 error norm"]}");
        }
       
        NUnit.Framework.Assert.LessOrEqual(errorData["l_1 error norm"], errThrsh,  
            $"l_1 error norm larger than allowed: {errorData["l_1 error norm"]}");

        NUnit.Framework.Assert.LessOrEqual(errorData["l_2 error norm"], errThrsh,  
            $"l_2 error norm larger than allowed: {errorData["l_2 error norm"]}");

        mode++;
    }

}

In [36]:
Plot2Ddata[,] Fig5 = new Plot2Ddata[1, 2];

//string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
var fmt = new PlotFormat();
fmt.Style = Styles.Lines; 
fmt.LineWidth = 2;
fmt.LineColor = LineColors.Black;


//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
var fig = GetAspectRatioOverTimeByModeDecomposition_Plot2Ddata(plotModeData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
var wnlt_dat = ReadPaperData("m2_Oh01_eta01_WNLT.txt", $"../paperData/AspectRatioOverTime/m2/", 2);
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = 0.9;
fig.YrangeMax = 1.2;
Fig5[0, 0] = fig; 

//fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta02-stagnantInit", "m2-Oh01-eta02-thirdOrderInit" });
fig = GetAspectRatioOverTimeByModeDecomposition_Plot2Ddata(plotModeData, new string[] { "m2-Oh01-eta02-stagnantInit", "m2-Oh01-eta02-thirdOrderInit" });
wnlt_dat = ReadPaperData("m2_Oh01_eta02_WNLT.txt", $"../paperData/AspectRatioOverTime/m2/", 2);
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = 0.8;
fig.YrangeMax = 1.4;
Fig5[0, 1] = fig;

Fig5.ToGnuplot().PlotSVG(xRes:1200,yRes:500)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.9 
 
 
 
 
 0.95 
 
 
 
 
 1 
 
 
 
 
 1.05 
 
 
 
 
 1.1 
 
 
 
 
 1.15 
 
 
 
 
 1.2 
 
 
 
 
 0 
 
 
 
 
 1 
 
 
 
 
 2 
 
 
 
 
 3 
 
 
 
 
 4 
 
 
 
 
 5 
 
 
 
 
 6 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L/W 
 
 
 
 
 t 
 
 
 
 
 XDG - stagnantInit 
 
 
 
 
 XDG - stagnantInit 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M249.3,62.1 L302.7,62.1 M86.1,96.3 L86.4,96.3 L86.8,96.3 L87.1,96.4 L87.4,96.5 L87.7,96.7
 L88.1,96.8 L88.4,97.1 L88.7,97.3 L89.0,97.6 L89.4,97.9 L89.7,98.3 L90.0,98.7 L90.4,99.1
 L90.7,99.6 L91.0,100.1 L91.3,100.6 L91.7,101.2 L92.0,101.8 L92.3,102.4 L92.6,103.1 L93.0,103.8
 L93.3,104.5 L93.6,105.3 L94.0,106.1 L94.3,106.9 L94.6,107.7 L94.9,108.6 L95.3,109.5 L95.6,110.5
 L95.9,111.4 L96.2,112.4 L96.6,113.5 L96.9,114.5 L97.2,115.6 L97.6,116.7 L97.9,117.8 L98.2,119.0
 L98.5,120.2 L98.9,121.4 L99.2,122.7 L99.5,124.0 L99.8,125.3 L100.2,126.6 L100.5,127.9 L100.8,129.3
 L101.1,130.7 L101.5,132.1 L101.8,133.6 L102.1,135.1 L102.5,136.6 L102.8,138.1 L103.1,139.7 L103.4,141.2
 L103.8,142.8 L104.1,144.4 L104.4,146.1 L104.7,147.7 L105.1,149.4 L105.4,151.1 L105.7,152.9 L106.1,154.6
 L106.4,156.4 L106.7,158.2 L107.0,160.0 L107.4,161.8 L107.7,163.7 L108.0,165.6 L108.3,167.5 L108.7,169.4
 L109.0,171.3 L109.3,173.3 L109.7,175.2 L110.0,177.2 L110.3,179.2 L110.6,181.2 L111.0,183.3 L111.3,185.3
 L111.6,187.4 L111.9,189.5 L112.3,191.6 L112.6,193.7 L112.9,195.8 L113.3,198.0 L113.6,200.1 L113.9,202.3
 L114.2,204.4 L114.6,206.6 L114.9,208.8 L115.2,211.0 L115.5,213.3 L115.9,215.5 L116.2,217.7 L116.5,220.0
 L116.9,222.2 L117.2,224.5 L117.5,226.8 L117.8,229.0 L118.2,231.3 L118.5,233.6 L118.8,235.9 L119.1,238.2
 L119.5,240.5 L119.8,242.8 L120.1,245.1 L120.5,247.4 L120.8,249.7 L121.1,252.0 L121.4,254.3 L121.8,256.6
 L122.1,258.9 L122.4,261.2 L122.7,263.6 L123.1,265.9 L123.4,268.2 L123.7,270.5 L124.0,272.8 L124.4,275.0
 L124.7,277.3 L125.0,279.6 L125.4,281.9 L125.7,284.1 L126.0,286.4 L126.3,288.7 L126.7,290.9 L127.0,293.2
 L127.3,295.4 L127.6,297.6 L128.0,299.8 L128.3,302.0 L128.6,304.2 L129.0,306.4 L129.3,308.6 L129.6,310.7
 L129.9,312.9 L130.3,315.0 L130.6,317.1 L130.9,319.2 L131.2,321.3 L131.6,323.4 L131.9,325.5 L132.2,327.5
 L132.6,329.6 L132.9,331.6 L133.2,333.6 L133.5,335.6 L133.9,337.5 L134.2,339.5 L134.5,341.4 L134.8,343.4
 L135.2,345.3 L135.5,347.1 L135.8,349.0 L136.2,350.9 L136.5,352.7 L136.8,354.5 L137.1,356.3 L137.5,358.1
 L137.8,359.8 L138.1,361.5 L138.4,363.3 L138.8,364.9 L139.1,366.6 L139.4,368.3 L139.8,369.9 L140.1,371.5
 L140.4,373.1 L140.7,374.6 L141.1,376.2 L141.4,377.7 L141.7,379.2 L142.0,380.6 L142.4,382.1 L142.7,383.5
 L143.0,384.9 L143.4,386.3 L143.7,387.6 L144.0,389.0 L144.3,390.3 L144.7,391.5 L145.0,392.8 L145.3,394.0
 L145.6,395.3 L146.0,396.4 L146.3,397.6 L146.6,398.7 L146.9,399.8 L147.3,400.9 L147.6,402.0 L147.9,403.0
 L148.3,404.1 L148.6,405.0 L148.9,406.0 L149.2,406.9 L149.6,407.9 L149.9,408.8 L150.2,409.6 L150.5,410.5
 L150.9,411.3 L151.2,412.1 L151.5,412.8 L151.9,413.6 L152.2,414.3 L152.5,415.0 L152.8,415.6 L153.2,416.3
 L153.5,416.9 L153.8,417.5 L154.1,418.1 L154.5,418.6 L154.8,419.1 L155.1,419.6 L155.5,420.1 L155.8,420.5
 L156.1,420.9 L156.4,421.3 L156.8,421.7 L157.1,422.1 L157.4,422.4 L157.7,422.7 L158.1,422.9 L158.4,423.2
 L158.7,423.4 L159.1,423.6 L159.4,423.8 L159.7,423.9 L160.0,424.0 L160.4,424.2 L160.7,424.2 L161.0,424.3
 L161.3,424.3 L161.7,424.3 L162.0,424.3 L162.3,424.3 L162.7,424.2 L163.0,424.1 L163.3,424.0 L163.6,423.9
 L164.0,423.7 L164.3,423.6 L164.6,423.4 L164.9,423.1 L165.3,422.9 L165.6,422.6 L165.9,422.4 L166.2,422.0
 L166.6,421.7 L166.9,421.4 L167.2,421.0 L167.6,420.6 L167.9,42

In [180]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m2-Oh01-eta01-stagnantInit", 0.003, 6, false, false);

checking mode 0:
checking mode 2:
checking mode 4:
checking mode 6:


In [182]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m2-Oh01-eta01-thirdOrderInit", 0.006, 6, false, false);

checking mode 0:
checking mode 2:
checking mode 4:
checking mode 6:


In [185]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m2-Oh01-eta02-stagnantInit", 0.001, 6, false, false);

checking mode 0:
checking mode 2:
checking mode 4:
checking mode 6:


In [187]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m2-Oh01-eta02-thirdOrderInit", 0.01, 6, false, false);

checking mode 0:
checking mode 2:
checking mode 4:
checking mode 6:


### FIG. 6.
Aspect ratio $L/W$ over time for $Oh = 0.1$, $m = 2$ and $\eta_0 = 0.4$ (case 1.3).

In [126]:
Plot2Ddata[,] Fig5 = new Plot2Ddata[1, 2];

//string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
var fmt = new PlotFormat();
fmt.Style = Styles.Lines;
fmt.DashType = DashTypes.Dashed;
fmt.LineWidth = 2;
// fmt.PointType = PointTypes.Circle;
// fmt.PointSize = 1;
fmt.LineColor = LineColors.Black;


//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta04-stagnantInit", "m2-Oh01-eta04-thirdOrderInit" }, 4.0);
var fig = GetAspectRatioOverTimeByModeDecomposition_Plot2Ddata(plotModeData, new string[] { "m2-Oh01-eta04-stagnantInit", "m2-Oh01-eta04-thirdOrderInit" }, 4.0);
var wnlt_dat = ReadPaperData("m2_Oh01_eta04_WNLT.txt", $"../paperData/AspectRatioOverTime/m2/", 2);
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = 0.5;
fig.YrangeMax = 2.7;
Fig5[0, 0] = fig; 

//fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta04-stagnantInit" });
fig = GetAspectRatioOverTimeByModeDecomposition_Plot2Ddata(plotModeData, new string[] { "m2-Oh01-eta04-stagnantInit" });
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
var fmt2 = fmt.CloneAs();
fmt2.LineColor = LineColors.Grey;
var meradji_dat = ReadPaperData("m2_Oh01_eta04_Meradji.txt", $"../paperData/AspectRatioOverTime/m2/", 1);
fig.AddDataGroup("Meradji et .al (2001)", meradji_dat.GetColumn(0), meradji_dat.GetColumn(1), fmt2);
var becker_dat = ReadPaperData("m2_Oh01_eta04_Becker.txt", $"../paperData/AspectRatioOverTime/m2/", 1);
fig.AddDataGroup("Becker (1994)", becker_dat.GetColumn(0), becker_dat.GetColumn(1), fmt2);
var basaran_dat = ReadPaperData("m2_Oh01_eta04_Basaran.txt", $"../paperData/AspectRatioOverTime/m2/", 1);
fig.AddDataGroup("Basaran (1992)", basaran_dat.GetColumn(0), basaran_dat.GetColumn(1), fmt2);
fig.YrangeMin = 0.6;
fig.YrangeMax = 1.8;
Fig5[0, 1] = fig;

Fig5.ToGnuplot().PlotSVG(xRes:1200,yRes:500)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L/W 
 
 
 
 
 t 
 
 
 
 
 XDG - stagnantInit 
 
 
 
 
 XDG - stagnantInit 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M249.3,62.1 L302.7,62.1 M77.8,206.4 L78.4,206.4 L79.0,206.5 L79.5,206.6 L80.1,206.7 L80.7,207.0
 L81.3,207.2 L81.9,207.5 L82.5,207.8 L83.0,208.2 L83.6,208.6 L84.2,209.0 L84.8,209.4 L85.4,209.9
 L86.0,210.4 L86.5,210.9 L87.1,211.4 L87.7,211.9 L88.3,212.5 L88.9,213.0 L89.5,213.6 L90.0,214.2
 L90.6,214.8 L91.2,215.4 L91.8,216.0 L92.4,216.7 L93.0,217.3 L93.5,217.9 L94.1,218.6 L94.7,219.2
 L95.3,219.9 L95.9,220.6 L96.5,221.2 L97.0,221.9 L97.6,222.6 L98.2,223.3 L98.8,224.0 L99.4,224.7
 L99.9,225.4 L100.5,226.1 L101.1,226.8 L101.7,227.5 L102.3,228.3 L102.9,229.0 L103.4,229.7 L104.0,230.5
 L104.6,231.2 L105.2,231.9 L105.8,232.7 L106.4,233.4 L106.9,234.2 L107.5,235.0 L108.1,235.7 L108.7,236.5
 L109.3,237.3 L109.9,238.0 L110.4,238.8 L111.0,239.6 L111.6,240.4 L112.2,241.1 L112.8,241.9 L113.4,242.7
 L113.9,243.5 L114.5,244.3 L115.1,245.1 L115.7,245.9 L116.3,246.7 L116.9,247.5 L117.4,248.3 L118.0,249.1
 L118.6,249.9 L119.2,250.7 L119.8,251.5 L120.3,252.4 L120.9,253.2 L121.5,254.0 L122.1,254.9 L122.7,255.7
 L123.3,256.6 L123.8,257.4 L124.4,258.3 L125.0,259.2 L125.6,260.0 L126.2,260.9 L126.8,261.8 L127.3,262.7
 L127.9,263.6 L128.5,264.6 L129.1,265.5 L129.7,266.4 L130.3,267.3 L130.8,268.3 L131.4,269.3 L132.0,270.2
 L132.6,271.2 L133.2,272.2 L133.8,273.2 L134.3,274.2 L134.9,275.2 L135.5,276.2 L136.1,277.3 L136.7,278.3
 L137.3,279.4 L137.8,280.4 L138.4,281.5 L139.0,282.6 L139.6,283.7 L140.2,284.8 L140.8,285.9 L141.3,287.0
 L141.9,288.1 L142.5,289.3 L143.1,290.4 L143.7,291.6 L144.2,292.7 L144.8,293.9 L145.4,295.1 L146.0,296.3
 L146.6,297.4 L147.2,298.6 L147.7,299.8 L148.3,301.0 L148.9,302.2 L149.5,303.5 L150.1,304.7 L150.7,305.9
 L151.2,307.1 L151.8,308.4 L152.4,309.6 L153.0,310.8 L153.6,312.1 L154.2,313.3 L154.7,314.6 L155.3,315.8
 L155.9,317.1 L156.5,318.3 L157.1,319.5 L157.7,320.8 L158.2,322.0 L158.8,323.3 L159.4,324.5 L160.0,325.7
 L160.6,327.0 L161.2,328.2 L161.7,329.4 L162.3,330.6 L162.9,331.8 L163.5,333.0 L164.1,334.2 L164.6,335.4
 L165.2,336.6 L165.8,337.8 L166.4,339.0 L167.0,340.1 L167.6,341.3 L168.1,342.5 L168.7,343.6 L169.3,344.7
 L169.9,345.9 L170.5,347.0 L171.1,348.1 L171.6,349.2 L172.2,350.3 L172.8,351.4 L173.4,352.5 L174.0,353.5
 L174.6,354.6 L175.1,355.6 L175.7,356.7 L176.3,357.7 L176.9,358.7 L177.5,359.7 L178.1,360.7 L178.6,361.7
 L179.2,362.7 L179.8,363.7 L180.4,364.6 L181.0,365.5 L181.6,366.5 L182.1,367.4 L182.7,368.3 L183.3,369.2
 L183.9,370.0 L184.5,370.9 L185.0,371.7 L185.6,372.6 L186.2,373.4 L186.8,374.2 L187.4,375.0 L188.0,375.8
 L188.5,376.6 L189.1,377.4 L189.7,378.1 L190.3,378.9 L190.9,379.6 L191.5,380.3 L192.0,381.0 L192.6,381.7
 L193.2,382.3 L193.8,382.9 L194.4,383.6 L195.0,384.1 L195.5,384.7 L196.1,385.2 L196.7,385.8 L197.3,386.3
 L197.9,386.8 L198.5,387.3 L199.0,387.8 L199.6,388.3 L200.2,388.8 L200.8,389.2 L201.4,389.7 L202.0,390.1
 L202.5,390.6 L203.1,391.0 L203.7,391.4 L204.3,391.8 L204.9,392.2 L205.4,392.6 L206.0,392.9 L206.6,393.3
 L207.2,393.6 L207.8,393.9 L208.4,394.2 L208.9,394.5 L209.5,394.8 L210.1,395.1 L210.7,395.4 L211.3,395.6
 L211.9,395.9 L212.4,396.1 L213.0,396.3 L213.6,396.5 L214.2,396.7 L214.8,396.9 L215.4,397.1 L215.9,397.2
 L216.5,397.4 L217.1,397.5 L217.7,397.6 L218.3,397.8 L218.9,397.9 L219.4,398.0 L220.0,398.0 L220.6,398.1
 L221.2,398.2 L221.8,398.2 L222.4,398.3 L222.9,398.3 L22

In [176]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m2-Oh01-eta04-stagnantInit", 0.008, 6, false, false);

checking mode 0:
checking mode 2:
checking mode 4:
checking mode 6:


In [178]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m2-Oh01-eta04-stagnantInit", 0.008, 6, false, false);

checking mode 0:
checking mode 2:
checking mode 4:
checking mode 6:


### FIG. 9.
Aspect ratio $L/W$ over time for $Oh = 0.1$ and $m = 3$. Left: $\eta_0 = 0.15$ (case 2.1). Right: $\eta_0 = 0.4$ (case 2.2).

In [39]:
Plot2Ddata[,] Fig9 = new Plot2Ddata[1, 2];

//string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
var fmt = new PlotFormat();
fmt.Style = Styles.Lines; 
fmt.LineWidth = 2;
fmt.LineColor = LineColors.Black;


//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
var fig = GetAspectRatioOverTimeByModeDecomposition_Plot2Ddata(plotModeData, new string[] { "m3-Oh01-eta015-stagnantInit", "m3-Oh01-eta015-thirdOrderInit" }, 4.0);
var wnlt_dat = ReadPaperData("m3_Oh01_eta015_WNLT.txt", $"../paperData/AspectRatioOverTime/m3/", 2);
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = 0.99;
fig.YrangeMax = 1.025;
Fig9[0, 0] = fig; 

//fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta02-stagnantInit", "m2-Oh01-eta02-thirdOrderInit" });
fig = GetAspectRatioOverTimeByModeDecomposition_Plot2Ddata(plotModeData, new string[] { "m3-Oh01-eta04-stagnantInit" }, 4.0);
wnlt_dat = ReadPaperData("m3_Oh01_eta04_WNLT.txt", $"../paperData/AspectRatioOverTime/m3/", 2);
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = 0.95;
fig.YrangeMax = 1.2;
Fig9[0, 1] = fig;

Fig9.ToGnuplot().PlotSVG(xRes:1200,yRes:500)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.99 
 
 
 
 
 0.995 
 
 
 
 
 1 
 
 
 
 
 1.005 
 
 
 
 
 1.01 
 
 
 
 
 1.015 
 
 
 
 
 1.02 
 
 
 
 
 1.025 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L/W 
 
 
 
 
 t 
 
 
 
 
 XDG - stagnantInit 
 
 
 
 
 XDG - stagnantInit 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M249.3,62.1 L302.7,62.1 M94.4,324.1 L95.0,324.0 L95.5,323.1 L96.1,321.3 L96.6,318.6 L97.2,315.2
 L97.8,311.0 L98.3,306.2 L98.9,300.8 L99.5,294.8 L100.0,288.5 L100.6,281.7 L101.1,274.6 L101.7,267.1
 L102.3,259.5 L102.8,251.6 L103.4,243.6 L104.0,235.4 L104.5,227.0 L105.1,218.7 L105.6,210.2 L106.2,202.0
 L106.8,193.8 L107.3,185.6 L107.9,177.5 L108.5,169.5 L109.0,161.6 L109.6,154.0 L110.1,146.6 L110.7,139.3
 L111.3,132.4 L111.8,125.7 L112.4,119.3 L113.0,113.2 L113.5,107.2 L114.1,101.6 L114.6,96.4 L115.2,91.4
 L115.8,86.8 L116.3,82.6 L116.9,78.6 L117.4,75.0 L118.0,71.7 L118.6,68.7 L119.1,66.0 L119.7,63.6
 L120.3,61.6 L120.8,59.7 L121.4,58.3 L121.9,57.0 L122.5,56.2 L123.1,55.7 L123.6,55.4 L124.2,55.3
 L124.8,55.6 L125.3,56.0 L125.9,56.6 L126.4,57.5 L127.0,58.6 L127.6,59.9 L128.1,61.4 L128.7,63.0
 L129.3,64.9 L129.8,66.8 L130.4,69.0 L130.9,71.2 L131.5,73.5 L132.1,76.0 L132.6,78.6 L133.2,81.3
 L133.7,84.1 L134.3,86.9 L134.9,89.8 L135.4,92.8 L136.0,96.3 L136.6,99.3 L137.1,102.1 L137.7,105.6
 L138.2,108.7 L138.8,111.4 L139.4,114.5 L139.9,117.6 L140.5,121.1 L141.1,123.8 L141.6,127.2 L142.2,129.8
 L142.7,133.1 L143.3,135.6 L143.9,138.4 L144.4,141.1 L145.0,144.1 L145.6,146.5 L146.1,148.9 L146.7,151.4
 L147.2,153.7 L147.8,155.9 L148.4,158.1 L148.9,160.1 L149.5,162.0 L150.1,163.9 L150.6,165.7 L151.2,167.4
 L151.7,168.9 L152.3,170.4 L152.9,171.8 L153.4,173.0 L154.0,174.1 L154.5,175.1 L155.1,176.0 L155.7,176.7
 L156.2,177.4 L156.8,178.0 L157.4,178.5 L157.9,178.8 L158.5,179.1 L159.0,179.4 L159.6,179.5 L160.2,179.5
 L160.7,179.5 L161.3,179.4 L161.9,179.2 L162.4,179.0 L163.0,178.7 L163.5,178.3 L164.1,177.9 L164.7,177.4
 L165.2,176.9 L165.8,176.3 L166.4,175.7 L166.9,175.1 L167.5,174.4 L168.0,173.7 L168.6,173.0 L169.2,172.2
 L169.7,171.5 L170.3,170.7 L170.8,169.9 L171.4,169.1 L172.0,168.4 L172.5,167.6 L173.1,166.8 L173.7,166.0
 L174.2,165.3 L174.8,164.6 L175.3,163.8 L175.9,163.1 L176.5,162.5 L177.0,161.8 L177.6,161.2 L178.2,160.6
 L178.7,160.0 L179.3,159.6 L179.8,159.0 L180.4,158.7 L181.0,158.3 L181.5,158.0 L182.1,157.7 L182.7,157.5
 L183.2,157.3 L183.8,157.3 L184.3,157.2 L184.9,157.1 L185.5,157.2 L186.0,157.3 L186.6,157.7 L187.2,157.8
 L187.7,158.0 L188.3,158.4 L188.8,158.8 L189.4,159.3 L190.0,160.2 L190.5,160.5 L191.1,161.1 L191.6,162.2
 L192.2,162.7 L192.8,163.5 L193.3,164.4 L193.9,165.3 L194.5,166.4 L195.0,167.5 L195.6,168.6 L196.1,170.4
 L196.7,171.2 L197.3,172.5 L197.8,173.9 L198.4,175.8 L199.0,177.3 L199.5,178.5 L200.1,180.0 L200.6,181.7
 L201.2,183.4 L201.8,185.2 L202.3,187.0 L202.9,188.8 L203.5,190.8 L204.0,192.7 L204.6,194.8 L205.1,196.8
 L205.7,198.9 L206.3,201.1 L206.8,203.2 L207.4,205.5 L207.9,208.2 L208.5,210.2 L209.1,212.4 L209.6,214.7
 L210.2,217.5 L210.8,219.9 L211.3,222.0 L211.9,224.3 L212.4,226.7 L213.0,229.1 L213.6,231.6 L214.1,234.4
 L214.7,236.6 L215.3,239.0 L215.8,241.4 L216.4,243.9 L216.9,246.3 L217.5,248.7 L218.1,251.2 L218.6,253.6
 L219.2,256.0 L219.8,258.4 L220.3,260.8 L220.9,263.2 L221.4,265.5 L222.0,267.9 L222.6,270.2 L223.1,272.5
 L223.7,274.7 L224.3,276.9 L224.8,279.4 L225.4,281.4 L225.9,283.5 L226.5,285.6 L227.1,287.6 L227.6,289.6
 L228.2,291.6 L228.7,293.5 L229.3,295.4 L229.9,297.5 L230.4,299.2 L231.0,300.9 L231.6,302.6 L232.

In [171]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m3-Oh01-eta015-stagnantInit", 0.03, 6, true, false);

checking mode 0:
checking mode 1:
checking mode 2:
checking mode 3:
checking mode 4:
checking mode 5:
checking mode 6:


In [172]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m3-Oh01-eta015-thirdOrderInit", 0.005, 6, true, false);

checking mode 0:
checking mode 1:
checking mode 2:
checking mode 3:
checking mode 4:
checking mode 5:
checking mode 6:


In [174]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m3-Oh01-eta04-stagnantInit", 0.2, 6, true, false);

checking mode 0:
checking mode 1:
checking mode 2:
checking mode 3:
checking mode 4:
checking mode 5:
checking mode 6:


### FIG. 10.
Aspect ratio $L/W$ over time for $Oh = 0.1$ and $m = 4$. Left: $\eta_0 = 0.1$ (case 3.1). Right: $\eta_0 = 0.4$ (case 3.2).

In [40]:
Plot2Ddata[,] Fig10 = new Plot2Ddata[1, 2];

//string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
var fmt = new PlotFormat();
fmt.Style = Styles.Lines; 
fmt.LineWidth = 2;
fmt.LineColor = LineColors.Black;


//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
var fig = GetAspectRatioOverTimeByModeDecomposition_Plot2Ddata(plotModeData, new string[] { "m4-Oh01-eta01-stagnantInit", "m4-Oh01-eta01-thirdOrderInit" }, 4.0);
var wnlt_dat = ReadPaperData("m4_Oh01_eta01_WNLT.txt", $"../paperData/AspectRatioOverTime/m4/", 2);
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = 0.96;
fig.YrangeMax = 1.08;
Fig10[0, 0] = fig; 

//fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta02-stagnantInit", "m2-Oh01-eta02-thirdOrderInit" });
fig = GetAspectRatioOverTimeByModeDecomposition_Plot2Ddata(plotModeData, new string[] { "m4-Oh01-eta04-stagnantInit" }, 4.0);
wnlt_dat = ReadPaperData("m4_Oh01_eta04_WNLT.txt", $"../paperData/AspectRatioOverTime/m4/", 2);
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = 0.9;
fig.YrangeMax = 1.4;
Fig10[0, 1] = fig;

Fig10.ToGnuplot().PlotSVG(xRes:1200,yRes:500)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.96 
 
 
 
 
 0.98 
 
 
 
 
 1 
 
 
 
 
 1.02 
 
 
 
 
 1.04 
 
 
 
 
 1.06 
 
 
 
 
 1.08 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L/W 
 
 
 
 
 t 
 
 
 
 
 XDG - stagnantInit 
 
 
 
 
 XDG - stagnantInit 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M249.3,62.1 L302.7,62.1 M86.1,106.1 L86.7,106.1 L87.2,106.3 L87.8,106.6 L88.4,107.1 L89.0,107.8
 L89.5,108.6 L90.1,109.5 L90.7,110.7 L91.3,112.0 L91.8,113.5 L92.4,115.1 L93.0,116.8 L93.5,118.8
 L94.1,120.9 L94.7,123.1 L95.3,125.5 L95.8,128.1 L96.4,130.8 L97.0,133.7 L97.6,136.8 L98.1,140.1
 L98.7,143.5 L99.3,147.1 L99.8,150.9 L100.4,154.9 L101.0,158.9 L101.6,163.2 L102.1,167.7 L102.7,172.3
 L103.3,177.1 L103.8,182.0 L104.4,187.1 L105.0,192.3 L105.6,197.7 L106.1,203.1 L106.7,208.7 L107.3,214.4
 L107.9,220.2 L108.4,226.0 L109.0,232.0 L109.6,237.9 L110.1,244.0 L110.7,250.0 L111.3,256.1 L111.9,262.1
 L112.4,268.2 L113.0,274.2 L113.6,280.2 L114.2,286.1 L114.7,292.0 L115.3,297.8 L115.9,303.4 L116.4,309.0
 L117.0,314.4 L117.6,319.7 L118.2,324.9 L118.7,329.9 L119.3,334.8 L119.9,339.5 L120.5,344.0 L121.0,348.3
 L121.6,352.5 L122.2,356.4 L122.7,360.1 L123.3,363.6 L123.9,366.8 L124.5,369.9 L125.0,372.7 L125.6,375.3
 L126.2,377.7 L126.7,379.8 L127.3,381.6 L127.9,383.3 L128.5,384.7 L129.0,385.9 L129.6,386.8 L130.2,387.4
 L130.8,387.9 L131.3,388.1 L131.9,388.1 L132.5,387.9 L133.0,387.4 L133.6,386.8 L134.2,386.0 L134.8,385.0
 L135.3,383.8 L135.9,382.4 L136.5,380.9 L137.1,379.2 L137.6,377.4 L138.2,375.4 L138.8,373.2 L139.3,371.0
 L139.9,368.6 L140.5,366.2 L141.1,363.6 L141.6,361.0 L142.2,358.3 L142.8,355.5 L143.4,352.6 L143.9,349.7
 L144.5,346.8 L145.1,343.8 L145.6,340.8 L146.2,337.7 L146.8,334.7 L147.4,331.6 L147.9,328.6 L148.5,325.5
 L149.1,322.4 L149.6,319.3 L150.2,316.3 L150.8,313.2 L151.4,310.2 L151.9,307.2 L152.5,304.2 L153.1,301.3
 L153.7,298.5 L154.2,295.6 L154.8,292.8 L155.4,290.1 L155.9,287.5 L156.5,284.9 L157.1,282.3 L157.7,279.9
 L158.2,277.4 L158.8,275.1 L159.4,272.7 L160.0,270.5 L160.5,268.3 L161.1,266.1 L161.7,264.1 L162.2,262.1
 L162.8,260.2 L163.4,258.4 L164.0,256.6 L164.5,254.9 L165.1,253.3 L165.7,251.8 L166.3,250.4 L166.8,249.0
 L167.4,247.7 L168.0,246.5 L168.5,245.4 L169.1,244.3 L169.7,243.3 L170.3,242.4 L170.8,241.6 L171.4,240.8
 L172.0,240.2 L172.5,239.6 L173.1,239.0 L173.7,238.6 L174.3,238.2 L174.8,237.9 L175.4,237.7 L176.0,237.5
 L176.6,237.5 L177.1,237.5 L177.7,237.5 L178.3,237.7 L178.8,237.9 L179.4,238.1 L180.0,238.5 L180.6,238.9
 L181.1,239.3 L181.7,239.9 L182.3,240.5 L182.9,241.1 L183.4,241.9 L184.0,242.6 L184.6,243.5 L185.1,244.4
 L185.7,245.3 L186.3,246.3 L186.9,247.4 L187.4,248.5 L188.0,249.6 L188.6,250.8 L189.2,252.1 L189.7,253.4
 L190.3,254.7 L190.9,256.1 L191.4,257.5 L192.0,258.9 L192.6,260.4 L193.2,261.9 L193.7,263.4 L194.3,265.0
 L194.9,266.6 L195.4,268.2 L196.0,269.8 L196.6,271.5 L197.2,273.1 L197.7,274.8 L198.3,276.5 L198.9,278.2
 L199.5,279.9 L200.0,281.6 L200.6,283.3 L201.2,284.9 L201.7,286.6 L202.3,288.3 L202.9,290.0 L203.5,291.6
 L204.0,293.3 L204.6,294.9 L205.2,296.5 L205.8,298.1 L206.3,299.7 L206.9,301.2 L207.5,302.7 L208.0,304.2
 L208.6,305.7 L209.2,307.1 L209.8,308.5 L210.3,309.9 L210.9,311.2 L211.5,312.5 L212.0,313.7 L212.6,314.9
 L213.2,316.1 L213.8,317.2 L214.3,318.3 L214.9,319.3 L215.5,320.3 L216.1,321.2 L216.6,322.1 L217.2,322.9
 L217.8,323.8 L218.3,324.6 L218.9,325.3 L219.5,325.9 L220.1,326.6 L220.6,327.1 L221.2,327.6 L221.8,328.1
 L222.4,328.5 L222.9,328.9 L223.5,329.2 L224.1,329.5 L224.6,329.8 L225.2,330.0 L225.8,330.1 L226.4,33

In [160]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m4-Oh01-eta01-stagnantInit", 0.008, 6, false, false);

checking mode 0:
checking mode 2:
checking mode 4:
checking mode 6:


In [162]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m4-Oh01-eta01-thirdOrderInit", 0.008, 6, false, false);

checking mode 0:
checking mode 2:
checking mode 4:
checking mode 6:


In [165]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m4-Oh01-eta04-stagnantInit", 0.04, 6, false, false);

checking mode 0:
checking mode 2:
checking mode 4:
checking mode 6:


### FIG. 14.
Aspect ratio $L/W$ over time for $Oh = 0.56$ and $m = 4$, and $\eta_0 = 0.05$. The linear theory predicts an aperiodic motion.

In [41]:
Plot2Ddata[,] Fig14 = new Plot2Ddata[1, 1];

//string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
var fmt = new PlotFormat();
fmt.Style = Styles.Lines; 
fmt.LineWidth = 2;
fmt.LineColor = LineColors.Black;


//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
var fig = GetAspectRatioOverTimeByModeDecomposition_Plot2Ddata(plotModeData, new string[] { "m4-Oh056-eta005-stagnantInit", "m4-Oh056-eta005-thirdOrderInit" }, 4.0);
var wnlt_dat = ReadPaperData("m4_Oh056_eta005_WNLT.txt", $"../paperData/AspectRatioOverTime/m4/", 2);
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = 0.97;
fig.YrangeMax = 1.04;
Fig14[0, 0] = fig; 

Fig14.ToGnuplot().PlotSVG(xRes:600,yRes:500)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.97 
 
 
 
 
 0.98 
 
 
 
 
 0.99 
 
 
 
 
 1 
 
 
 
 
 1.01 
 
 
 
 
 1.02 
 
 
 
 
 1.03 
 
 
 
 
 1.04 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L/W 
 
 
 
 
 t 
 
 
 
 
 XDG - stagnantInit 
 
 
 
 
 XDG - stagnantInit 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M249.3,62.1 L302.7,62.1 M86.1,93.9 L86.7,93.9 L87.2,94.1 L87.8,94.5 L88.4,95.0 L89.0,95.8
 L89.5,96.7 L90.1,97.7 L90.7,98.9 L91.3,100.2 L91.8,101.6 L92.4,103.1 L93.0,104.7 L93.5,106.4
 L94.1,108.2 L94.7,110.0 L95.3,111.9 L95.8,113.8 L96.4,115.8 L97.0,117.9 L97.6,119.9 L98.1,122.0
 L98.7,124.2 L99.3,126.3 L99.8,128.5 L100.4,130.7 L101.0,132.9 L101.6,135.1 L102.1,137.3 L102.7,139.5
 L103.3,141.7 L103.8,144.0 L104.4,146.2 L105.0,148.3 L105.6,150.5 L106.1,152.7 L106.7,154.9 L107.3,157.0
 L107.9,159.1 L108.4,161.2 L109.0,163.3 L109.6,165.4 L110.1,167.4 L110.7,169.4 L111.3,171.4 L111.9,173.4
 L112.4,175.3 L113.0,177.3 L113.6,179.2 L114.2,181.0 L114.7,182.9 L115.3,184.7 L115.9,186.5 L116.4,188.2
 L117.0,189.9 L117.6,191.6 L118.2,193.3 L118.7,194.9 L119.3,196.5 L119.9,198.1 L120.5,199.7 L121.0,201.2
 L121.6,202.7 L122.2,204.1 L122.7,205.6 L123.3,207.0 L123.9,208.4 L124.5,209.7 L125.0,211.0 L125.6,212.3
 L126.2,213.6 L126.7,214.9 L127.3,216.1 L127.9,217.3 L128.5,218.4 L129.0,219.6 L129.6,220.7 L130.2,221.8
 L130.8,222.8 L131.3,223.9 L131.9,224.9 L132.5,225.9 L133.0,226.9 L133.6,227.8 L134.2,228.8 L134.8,229.7
 L135.3,230.6 L135.9,231.4 L136.5,232.3 L137.1,233.1 L137.6,233.9 L138.2,234.7 L138.8,235.5 L139.3,236.2
 L139.9,237.0 L140.5,237.7 L141.1,238.4 L141.6,239.1 L142.2,239.7 L142.8,240.4 L143.4,241.0 L143.9,241.6
 L144.5,242.2 L145.1,242.8 L145.6,243.4 L146.2,244.0 L146.8,244.5 L147.4,245.0 L147.9,245.6 L148.5,246.1
 L149.1,246.6 L149.6,247.0 L150.2,247.5 L150.8,248.0 L151.4,248.4 L151.9,248.9 L152.5,249.3 L153.1,249.7
 L153.7,250.1 L154.2,250.5 L154.8,250.9 L155.4,251.3 L155.9,251.6 L156.5,252.0 L157.1,252.3 L157.7,252.7
 L158.2,253.0 L158.8,253.3 L159.4,253.6 L160.0,253.9 L160.5,254.2 L161.1,254.5 L161.7,254.8 L162.2,255.1
 L162.8,255.4 L163.4,255.6 L164.0,255.9 L164.5,256.2 L165.1,256.4 L165.7,256.6 L166.3,256.9 L166.8,257.1
 L167.4,257.3 L168.0,257.5 L168.5,257.8 L169.1,258.0 L169.7,258.2 L170.3,258.4 L170.8,258.6 L171.4,258.8
 L172.0,258.9 L172.5,259.1 L173.1,259.3 L173.7,259.5 L174.3,259.6 L174.8,259.8 L175.4,260.0 L176.0,260.1
 L176.6,260.3 L177.1,260.4 L177.7,260.6 L178.3,260.7 L178.8,260.8 L179.4,261.0 L180.0,261.1 L180.6,261.2
 L181.1,261.4 L181.7,261.5 L182.3,261.6 L182.9,261.7 L183.4,261.9 L184.0,262.0 L184.6,262.1 L185.1,262.2
 L185.7,262.3 L186.3,262.4 L186.9,262.5 L187.4,262.6 L188.0,262.7 L188.6,262.8 L189.2,262.9 L189.7,263.0
 L190.3,263.1 L190.9,263.2 L191.4,263.2 L192.0,263.3 L192.6,263.4 L193.2,263.5 L193.7,263.6 L194.3,263.7
 L194.9,263.7 L195.4,263.8 L196.0,263.9 L196.6,264.0 L197.2,264.0 L197.7,264.1 L198.3,264.2 L198.9,264.2
 L199.5,264.3 L200.0,264.4 L200.6,264.4 L201.2,264.5 L201.7,264.5 L202.3,264.6 L202.9,264.7 L203.5,264.7
 L204.0,264.8 L204.6,264.8 L205.2,264.9 L205.8,264.9 L206.3,265.0 L206.9,265.0 L207.5,265.1 L208.0,265.1
 L208.6,265.2 L209.2,265.2 L209.8,265.3 L210.3,265.3 L210.9,265.4 L211.5,265.4 L212.0,265.5 L212.6,265.5
 L213.2,265.5 L213.8,265.6 L214.3,265.6 L214.9,265.7 L215.5,265.7 L216.1,265.7 L216.6,265.8 L217.2,265.8
 L217.8,265.9 L218.3,265.9 L218.9,265.9 L219.5,266.0 L220.1,266.0 L220.6,266.0 L221.2,266.1 L221.8,266.1
 L222.4,266.1 L222.9,266.2 L223.5,266.2 L224.1,266.2 L224.6,266.3 L225.2,266.3 L225.8

In [156]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m4-Oh056-eta005-stagnantInit", 0.01, 6, false, false);

checking mode 0:
checking mode 2:
checking mode 4:
checking mode 6:


In [157]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m4-Oh056-eta005-thirdOrderInit", 0.003, 6, false, false);

checking mode 0:
checking mode 2:
checking mode 4:
checking mode 6:


### mode decompositon

In [42]:
public static Plot2Ddata GetModeDecompositionOverTime_Plot2Ddata(List<Plot2Ddata> modeDataS, int mode, string[] studies) {

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "t";
    plt.Ylabel = $"a_{mode}";

    int lcIdx = 0;
    for (int i = 0; i < studies.Length; i++) {
        var datPlt = modeDataS.Where(plt => plt.Title.Contains(studies[i])).Single();

        double[] modeValues = datPlt.dataGroups[mode].Values.CloneAs();
        if (mode == 0) {
            // shift mode 0 by 1.0
            for (int j = 0; j < modeValues.Length; j++) {
                modeValues[j] = modeValues[j] - 1.0;
            }
        }
    
        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines;
        fmt.DashType = studies[i].Contains("stagnantInit") ? DashTypes.Solid : DashTypes.DotDashed;
        fmt.LineWidth = 2;
        LineColors[] customColors = new LineColors[] { LineColors.Blue, LineColors.Red, LineColors.Green };
        fmt.LineColor = GetCustomLineColorFromIndex(lcIdx, customColors);

        string[] nameParts = datPlt.Title.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        plt.AddDataGroup($"XDG - {nameParts[4]}", datPlt.dataGroups[0].Abscissas, modeValues, fmt);

        lcIdx++;      
    }   

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 4.0;

    plt.ShowLegend = true;

    return plt;        
}

### Fig 15.
Mode decomposition of the drop surface as a function of time for $Oh = 0.56$, $m = 4$, and $\eta_0 =
0.05$, represented by the coefficients of the series expansion of the drop surface shape.

In [44]:
Plot2Ddata[,] Fig15= new Plot2Ddata[2, 2];

//string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
var fmt = new PlotFormat();
fmt.Style = Styles.Lines; 
fmt.DashType = DashTypes.Dashed;
fmt.LineWidth = 2;
fmt.LineColor = LineColors.Black;


//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
var fig = GetModeDecompositionOverTime_Plot2Ddata(plotModeData, 0, new string[] { "m4-Oh056-eta005-stagnantInit", "m4-Oh056-eta005-thirdOrderInit"  });
var wnlt_dat = ReadPaperData("m4_Oh056_eta005 (0, 0)-WNLT.txt", $"../paperData/modeDecomposition/", 2);
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = -0.3e-3;
fig.YrangeMax = 0.1e-3;
fig.LegendAlignment = new string[] { "i", "b", "r"};
Fig15[0, 0] = fig; 

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
fig = GetModeDecompositionOverTime_Plot2Ddata(plotModeData, 2, new string[] { "m4-Oh056-eta005-stagnantInit", "m4-Oh056-eta005-thirdOrderInit"  });
wnlt_dat = ReadPaperData("m4_Oh056_eta005 (2, 0)-WNLT.txt", $"../paperData/modeDecomposition/", 2);
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = -25.0e-3;
fig.YrangeMax = 5.0e-3;
fig.LegendAlignment = new string[] { "i", "b", "r"};
Fig15[0, 1] = fig; 

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
fig = GetModeDecompositionOverTime_Plot2Ddata(plotModeData, 4, new string[] { "m4-Oh056-eta005-stagnantInit", "m4-Oh056-eta005-thirdOrderInit"  });
wnlt_dat = ReadPaperData("m4_Oh056_eta005 (4, 0)-WNLT.txt", $"../paperData/modeDecomposition/", 2);
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = -20.0e-3;
fig.YrangeMax = 60.0e-3;
fig.LegendAlignment = new string[] { "i", "t", "r"};
Fig15[1, 0] = fig; 

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
fig = GetModeDecompositionOverTime_Plot2Ddata(plotModeData, 6, new string[] { "m4-Oh056-eta005-stagnantInit", "m4-Oh056-eta005-thirdOrderInit"  });
wnlt_dat = ReadPaperData("m4_Oh056_eta005 (6, 0)-WNLT.txt", $"../paperData/modeDecomposition/", 2);
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = -1.0e-3;
fig.YrangeMax = 3.0e-3;
fig.LegendAlignment = new string[] { "i", "t", "r"};
Fig15[1, 1] = fig; 


Fig15.ToGnuplot().PlotSVG(xRes:1400,yRes:900)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.0003 
 
 
 
 
 -0.00025 
 
 
 
 
 -0.0002 
 
 
 
 
 -0.00015 
 
 
 
 
 -0.0001 
 
 
 
 
 -5x10 -5 
 
 
 
 
 0 
 
 
 
 
 5x10 -5 
 
 
 
 
 0.0001 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 a 0 
 
 
 
 
 t 
 
 
 
 
 XDG - stagnantInit 
 
 
 
 
 XDG - stagnantInit 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M348.3,319.0 L401.7,319.0 M112.0,369.9 L112.7,370.1 L113.3,369.2 L114.0,367.8 L114.7,365.8 L115.3,363.3
 L116.0,360.3 L116.6,356.9 L117.3,353.1 L118.0,349.0 L118.6,344.6 L119.3,340.0 L120.0,335.2 L120.6,330.3
 L121.3,325.2 L122.0,320.0 L122.6,314.8 L123.3,309.5 L123.9,304.2 L124.6,298.9 L125.3,293.6 L125.9,288.3
 L126.6,283.0 L127.3,277.8 L127.9,272.7 L128.6,267.6 L129.3,262.6 L129.9,257.7 L130.6,252.9 L131.3,248.2
 L131.9,243.6 L132.6,239.1 L133.2,234.7 L133.9,230.4 L134.6,226.3 L135.2,222.2 L135.9,218.3 L136.6,214.5
 L137.2,210.8 L137.9,207.2 L138.6,203.8 L139.2,200.4 L139.9,197.2 L140.5,194.1 L141.2,191.1 L141.9,188.2
 L142.5,185.4 L143.2,182.8 L143.9,180.2 L144.5,177.7 L145.2,175.4 L145.9,173.1 L146.5,170.9 L147.2,168.8
 L147.8,166.8 L148.5,164.9 L149.2,163.1 L149.8,161.3 L150.5,159.6 L151.2,158.0 L151.8,156.5 L152.5,155.0
 L153.2,153.6 L153.8,152.2 L154.5,151.0 L155.2,149.7 L155.8,148.6 L156.5,147.5 L157.1,146.4 L157.8,145.4
 L158.5,144.4 L159.1,143.5 L159.8,142.7 L160.5,141.8 L161.1,141.0 L161.8,140.3 L162.5,139.6 L163.1,138.9
 L163.8,138.2 L164.4,137.6 L165.1,137.0 L165.8,136.5 L166.4,136.0 L167.1,135.5 L167.8,135.0 L168.4,134.5
 L169.1,134.1 L169.8,133.7 L170.4,133.3 L171.1,133.0 L171.7,132.6 L172.4,132.3 L173.1,132.0 L173.7,131.7
 L174.4,131.4 L175.1,131.1 L175.7,130.9 L176.4,130.6 L177.1,130.4 L177.7,130.2 L178.4,130.0 L179.1,129.8
 L179.7,129.6 L180.4,129.4 L181.0,129.3 L181.7,129.1 L182.4,129.0 L183.0,128.8 L183.7,128.7 L184.4,128.6
 L185.0,128.5 L185.7,128.4 L186.4,128.3 L187.0,128.2 L187.7,128.1 L188.3,128.0 L189.0,127.9 L189.7,127.8
 L190.3,127.7 L191.0,127.7 L191.7,127.6 L192.3,127.5 L193.0,127.5 L193.7,127.4 L194.3,127.4 L195.0,127.3
 L195.6,127.2 L196.3,127.2 L197.0,127.2 L197.6,127.1 L198.3,127.1 L199.0,127.0 L199.6,127.0 L200.3,127.0
 L201.0,126.9 L201.6,126.9 L202.3,126.9 L203.0,126.9 L203.6,126.8 L204.3,126.8 L204.9,126.8 L205.6,126.8
 L206.3,126.7 L206.9,126.7 L207.6,126.7 L208.3,126.7 L208.9,126.7 L209.6,126.7 L210.3,126.6 L210.9,126.6
 L211.6,126.6 L212.2,126.6 L212.9,126.6 L213.6,126.6 L214.2,126.6 L214.9,126.6 L215.6,126.5 L216.2,126.5
 L216.9,126.5 L217.6,126.5 L218.2,126.5 L218.9,126.5 L219.5,126.5 L220.2,126.5 L220.9,126.5 L221.5,126.5
 L222.2,126.5 L222.9,126.5 L223.5,126.4 L224.2,126.4 L224.9,126.4 L225.5,126.4 L226.2,126.4 L226.9,126.4
 L227.5,126.4 L228.2,126.4 L228.8,126.4 L229.5,126.4 L230.2,126.4 L230.8,126.4 L231.5,126.4 L232.2,126.4
 L232.8,126.4 L233.5,126.4 L234.2,126.4 L234.8,126.4 L235.5,126.4 L236.1,126.4 L236.8,126.4 L237.5,126.4
 L238.1,126.4 L238.8,126.4 L239.5,126.4 L240.1,126.4 L240.8,126.4 L241.5,126.4 L242.1,126.4 L242.8,126.4
 L243.4,126.3 L244.1,126.3 L244.8,126.3 L245.4,126.3 L246.1,126.3 L246.8,126.3 L247.4,126.3 L248.1,126.3
 L248.8,126.3 L249.4,126.3 L250.1,126.3 L250.7,126.3 L251.4,126.3 L252.1,126.3 L252.7,126.3 L253.4,126.3
 L254.1,126.3 L254.7,126.3 L255.4,126.3 L256.1,126.3 L256.7,126.3 L257.4,126.3 L258.1,126.3 L258.7,126.3
 L259.4,126.3 L260.0,126.3 L260.7,126.3 L261.4,126.3 L262.0,126.3 L262.7,126.3 L263.4,126.3 L264.0,126.3
 L264.7,126.3 L265.4,126.3 L266.0,126.3 L266.7,126.3 L267.3,126.3 L268.0,126.3 L268.7,126.3 L269.3,126.

### north pole position $r_s(0,t)$ over time - by mode decomposition

In [45]:
public static double[] GetNorthPoleByModeDecomposition(Plot2Ddata modeData) {

    int numTimesteps = modeData.dataGroups[0].Abscissas.Length;
    double[] northPolePositions = new double[numTimesteps];

    for (int ts = 0; ts < numTimesteps; ts++) {

        double northPole = 0.0;

        int maxL = modeData.dataGroups.Count();
        for (int m = 0; m < maxL; m++) {

            if (m == 1)
                continue;

            double mValue = modeData.dataGroups[m].Values[ts];

            northPole += mValue * SphericalHarmonics.MyLegendre(m,0,Math.Cos(0.0));        
        }
        northPolePositions[ts] = northPole;
    }

    return northPolePositions;
}

In [46]:
public static Plot2Ddata GetNorthPoleOverTimeByModeDecomposition_Plot2Ddata(List<Plot2Ddata> modeDataS, string[] studies) {

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "t";
    plt.Ylabel = "r_s(0,t)";

    int lcIdx = 0;
    for (int i = 0; i < studies.Length; i++) {
        var datPlt = modeDataS.Where(plt => plt.Title.Contains(studies[i])).Single();

        double[] northPole = GetNorthPoleByModeDecomposition(datPlt);
    
        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines;
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        LineColors[] customColors = new LineColors[] { LineColors.Blue, LineColors.Red, LineColors.Green };
        fmt.LineColor = GetCustomLineColorFromIndex(lcIdx, customColors);

        //string[] nameParts = datPlt.Title.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        plt.AddDataGroup($"XDG", datPlt.dataGroups[0].Abscissas, northPole, fmt);

        lcIdx++;      
    }   

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 7.0;

    plt.ShowLegend = true;

    return plt;        
}

### FIG. 24.
Droplet north pole position $r_s(0,t)$ over time $t$ for drops with $Oh = 0.1$ — comparison of the
WNLT predictions with the XDG results: (a) $m = 2$ with $ \eta_0 = 0.4$, (b) $m = 3$ with $\eta_0 = 0.15$, and (c) $m = 4$
with $\eta_0 = 0.1$.

In [48]:
Plot2Ddata[,] Fig24 = new Plot2Ddata[2, 2];

//string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
var fmt = new PlotFormat();
fmt.Style = Styles.Lines; 
fmt.DashType = DashTypes.Dashed;
fmt.LineWidth = 2;
fmt.LineColor = LineColors.Black;


//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
var fig = GetNorthPoleOverTimeByModeDecomposition_Plot2Ddata(plotModeData, new string[] { "m2-Oh01-eta04-stagnantInit" });
var wnlt_dat = ReadPaperData("m2_Oh01_eta04_WNLT_northPole.dat", $"../paperData/NorthPoleOverTime/m2/");
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = 0.7;
fig.YrangeMax = 1.4;
Fig24[0, 0] = fig; 

//fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta02-stagnantInit", "m2-Oh01-eta02-thirdOrderInit" });
fig = GetNorthPoleOverTimeByModeDecomposition_Plot2Ddata(plotModeData, new string[] { "m3-Oh01-eta015-stagnantInit" });
wnlt_dat = ReadPaperData("m3_Oh01_eta015_WNLT_northPole.dat", $"../paperData/NorthPoleOverTime/m3/");
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = 0.9;
fig.YrangeMax = 1.15;
Fig24[0, 1] = fig;

//fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta02-stagnantInit", "m2-Oh01-eta02-thirdOrderInit" });
fig = GetNorthPoleOverTimeByModeDecomposition_Plot2Ddata(plotModeData, new string[] { "m4-Oh01-eta01-stagnantInit" });
wnlt_dat = ReadPaperData("m4_Oh01_eta01_WNLT_northPole.dat", $"../paperData/NorthPoleOverTime/m4/");
fig.AddDataGroup("WNLT", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = 0.95;
fig.YrangeMax = 1.1;
Fig24[1, 0] = fig;

Fig24.ToGnuplot().PlotSVG(xRes:1200,yRes:1000)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.7 
 
 
 
 
 0.8 
 
 
 
 
 0.9 
 
 
 
 
 1 
 
 
 
 
 1.1 
 
 
 
 
 1.2 
 
 
 
 
 1.3 
 
 
 
 
 1.4 
 
 
 
 
 0 
 
 
 
 
 1 
 
 
 
 
 2 
 
 
 
 
 3 
 
 
 
 
 4 
 
 
 
 
 5 
 
 
 
 
 6 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 r s (0,t) 
 
 
 
 
 t 
 
 
 
 
 XDG 
 
 
 
 
 XDG 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M426.9,62.1 L480.3,62.1 M77.8,59.9 L78.1,59.9 L78.5,59.9 L78.8,59.9 L79.1,60.0 L79.5,60.0
 L79.8,60.1 L80.1,60.2 L80.5,60.3 L80.8,60.4 L81.1,60.5 L81.5,60.6 L81.8,60.7 L82.1,60.9
 L82.5,61.1 L82.8,61.2 L83.1,61.4 L83.5,61.6 L83.8,61.8 L84.1,62.0 L84.5,62.3 L84.8,62.5
 L85.1,62.8 L85.5,63.0 L85.8,63.3 L86.1,63.6 L86.5,63.9 L86.8,64.2 L87.1,64.5 L87.5,64.8
 L87.8,65.1 L88.1,65.4 L88.5,65.8 L88.8,66.2 L89.1,66.5 L89.5,66.9 L89.8,67.3 L90.1,67.7
 L90.5,68.1 L90.8,68.6 L91.1,69.0 L91.5,69.5 L91.8,69.9 L92.1,70.4 L92.5,70.9 L92.8,71.4
 L93.1,71.9 L93.5,72.4 L93.8,73.0 L94.1,73.5 L94.5,74.1 L94.8,74.6 L95.1,75.2 L95.5,75.8
 L95.8,76.4 L96.1,77.0 L96.5,77.7 L96.8,78.3 L97.1,79.0 L97.5,79.6 L97.8,80.3 L98.1,81.0
 L98.5,81.7 L98.8,82.4 L99.1,83.1 L99.4,83.9 L99.8,84.6 L100.1,85.4 L100.4,86.1 L100.8,86.9
 L101.1,87.7 L101.4,88.6 L101.8,89.4 L102.1,90.3 L102.4,91.2 L102.8,92.1 L103.1,93.0 L103.4,93.9
 L103.8,94.9 L104.1,95.9 L104.4,96.9 L104.8,97.9 L105.1,99.0 L105.4,100.0 L105.8,101.1 L106.1,102.3
 L106.4,103.4 L106.8,104.6 L107.1,105.9 L107.4,107.1 L107.8,108.4 L108.1,109.7 L108.4,111.1 L108.8,112.5
 L109.1,113.9 L109.4,115.4 L109.8,116.9 L110.1,118.4 L110.4,120.0 L110.8,121.6 L111.1,123.2 L111.4,124.9
 L111.8,126.6 L112.1,128.3 L112.4,130.1 L112.8,131.9 L113.1,133.8 L113.4,135.7 L113.8,137.6 L114.1,139.6
 L114.4,141.6 L114.8,143.6 L115.1,145.7 L115.4,147.8 L115.8,149.9 L116.1,152.1 L116.4,154.3 L116.8,156.5
 L117.1,158.7 L117.4,161.0 L117.8,163.3 L118.1,165.7 L118.4,168.1 L118.8,170.5 L119.1,172.9 L119.4,175.4
 L119.8,177.8 L120.1,180.4 L120.4,182.9 L120.8,185.4 L121.1,188.0 L121.4,190.6 L121.8,193.2 L122.1,195.8
 L122.4,198.4 L122.8,201.0 L123.1,203.7 L123.4,206.4 L123.8,209.1 L124.1,211.8 L124.4,214.5 L124.8,217.2
 L125.1,219.8 L125.4,222.5 L125.8,225.2 L126.1,227.9 L126.4,230.6 L126.8,233.2 L127.1,235.9 L127.4,238.5
 L127.8,241.2 L128.1,243.9 L128.4,246.5 L128.8,249.2 L129.1,251.8 L129.4,254.4 L129.8,257.0 L130.1,259.6
 L130.4,262.2 L130.8,264.8 L131.1,267.4 L131.4,269.9 L131.8,272.4 L132.1,275.0 L132.4,277.5 L132.8,279.9
 L133.1,282.4 L133.4,284.9 L133.8,287.3 L134.1,289.7 L134.4,292.1 L134.8,294.4 L135.1,296.8 L135.4,299.1
 L135.8,301.4 L136.1,303.6 L136.4,305.9 L136.8,308.1 L137.1,310.3 L137.4,312.4 L137.8,314.5 L138.1,316.6
 L138.4,318.6 L138.8,320.6 L139.1,322.6 L139.4,324.5 L139.8,326.5 L140.1,328.5 L140.4,330.4 L140.8,332.3
 L141.1,334.1 L141.4,335.9 L141.7,337.7 L142.1,339.5 L142.4,341.2 L142.7,342.9 L143.1,344.5 L143.4,346.2
 L143.7,347.7 L144.1,349.1 L144.4,350.5 L144.7,351.7 L145.1,353.0 L145.4,354.2 L145.7,355.4 L146.1,356.6
 L146.4,357.7 L146.7,358.9 L147.1,360.0 L147.4,361.1 L147.7,362.2 L148.1,363.2 L148.4,364.3 L148.7,365.3
 L149.1,366.3 L149.4,367.2 L149.7,368.1 L150.1,369.1 L150.4,369.9 L150.7,370.8 L151.1,371.6 L151.4,372.4
 L151.7,373.1 L152.1,373.9 L152.4,374.6 L152.7,375.2 L153.1,375.9 L153.4,376.5 L153.7,377.1 L154.1,377.6
 L154.4,378.1 L154.7,378.6 L155.1,379.1 L155.4,379.5 L155.7,380.0 L156.1,380.4 L156.4,380.7 L156.7,381.1
 L157.1,381.4 L157.4,381.7 L157.7,381.9 L158.1,382.2 L158.4,382.4 L158.7,382.6 L159.1,382.7 L159.4,382.9
 L159.7,383.0 L160.1,383.1 L160.4,383.2 L160.7,383.3 L161.1,383.3 L161.4,383.4 L161.7,383.4 L162.1,383.3
 L162.4,383.3 L162.7,383.2 L163.1,383.1 L1